# UiO-66 replacement pilot — private saved batch
User explicitly approved this replacement: same512+16,2T4,max4.2h; execution receipt must confirm approval.
After explicit approval, freeze this notebook/code/512-pose manifest/checkpoint hashes
with a saved-batch execution receipt and a reviewed two-T4 deadline.
Use Save Version → Save & Run All; verify a numbered private version actually starts.
This one cell waits for the controller, then archives raw results and failures.
Download and hash-verify saved version outputs before declaring evidence secured.
No refit, new poses, 64-design expansion or DFT.


In [ ]:
import hashlib, json, traceback, zipfile
from datetime import datetime, timezone
from pathlib import Path
WORKING=Path('/kaggle/working')
workflow={'status':'started','started_utc':datetime.now(timezone.utc).isoformat()}
try:
    import hashlib, json, os, shutil, subprocess, sys, zipfile
    from datetime import datetime, timezone
    from pathlib import Path
    release_paths=list(Path('/kaggle/input').rglob('execution_release.json'))
    assert len(release_paths)==1, 'Attach the private authorized pilot release'
    AUTH=json.loads(release_paths[0].read_text())
    assert AUTH.get('execution_mode')=='saved_batch' and AUTH.get('replacement_pilot_approved') is True, 'Replacement run approval and saved-batch receipt required'
    bundles=list(Path('/kaggle/input').rglob('UIO66_Sampling_Pilot_v2_authorized.bin'))
    assert len(bundles)==1
    BUNDLE=bundles[0]
    assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()==AUTH['bundle_sha256']
    REPO=Path('/kaggle/working/MOF_sampling_authorized_portable')
    assert not REPO.exists(), 'Do not overwrite a previous extraction'
    with zipfile.ZipFile(BUNDLE) as archive:
        for name in archive.namelist():
            assert (REPO/name).resolve().is_relative_to(REPO.resolve())
        archive.extractall(REPO)
    manifest=json.loads((REPO/'bundle_manifest.json').read_text())['sha256']
    assert manifest==AUTH['file_sha256']
    for name,digest in manifest.items():
        file=REPO/name
        assert hashlib.sha256(file.read_bytes()).hexdigest()==digest,name
        file.chmod(0o444)
    os.chdir(REPO)
    ENV=dict(os.environ,PYTHONPATH=str(REPO),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',PYTHONDONTWRITEBYTECODE='1')
    ALLOCATION_START_UTC=AUTH['allocation_budget_start_utc']
    PRE=Path('/kaggle/working/uio66_sampling_preflight_portable')
    assert not PRE.exists()
    PRE.mkdir()
    gpu_info=subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.total,uuid','--format=csv,noheader'],text=True)
    assert len(gpu_info.strip().splitlines())==2 and all('T4' in r for r in gpu_info.strip().splitlines())
    (PRE/'allocation_observation.json').write_text(json.dumps({'gpu_observed_utc':datetime.now(timezone.utc).isoformat(),
     'budget_start_utc':ALLOCATION_START_UTC,'basis':'conservative timestamp frozen before session start; includes setup',
     'gpu_info':gpu_info,'release_sha256':hashlib.sha256(release_paths[0].read_bytes()).hexdigest()},indent=2)+chr(10))
    print('FROZEN RELEASE VERIFIED:',AUTH['bundle_sha256'])
    print(gpu_info)
    with (PRE/'install.log').open('w') as log:
        remaining=15120-(datetime.now(timezone.utc)-datetime.fromisoformat(ALLOCATION_START_UTC)).total_seconds()
        subprocess.run([sys.executable,'-m','pip','install','-q','fairchem-core==2.23.0','ase==3.26.0'],
            stdout=log,stderr=subprocess.STDOUT,check=True,timeout=max(1,remaining))
    subprocess.run([sys.executable,'-m','scripts.run_uio66_sampling_pilot','--output',str(PRE/'dry'/'report.json')],env=ENV,check=True)
    print('GPU-environment bundle/512-pose geometry preflight passed; no model loaded yet')

    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import hf_hub_download
    os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
    plan=json.loads((REPO/'data/design/uio66_sampling_pilot_v2.json').read_text())
    CHECKPOINT=Path(hf_hub_download('facebook/UMA','checkpoints/uma-s-1p2p1.pt',
     revision=plan['checkpoint_revision'],token=os.environ['HF_TOKEN']))
    digest=hashlib.sha256()
    with CHECKPOINT.open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''): digest.update(block)
    assert digest.hexdigest()==plan['checkpoint_sha256']==AUTH['checkpoint_sha256']
    (PRE/'checkpoint_verification.json').write_text(json.dumps({'sha256':digest.hexdigest(),
     'revision':plan['checkpoint_revision'],'verified_utc':datetime.now(timezone.utc).isoformat()},indent=2)+chr(10))
    print('PINNED CHECKPOINT VERIFIED:',digest.hexdigest())

    OUT=Path('/kaggle/working/uio66_sampling_authorized')
    assert not OUT.exists(), 'New output root required'
    controller_log=Path('/kaggle/working/uio66_sampling_controller.log')
    with controller_log.open('w') as log:
        CONTROLLER=subprocess.Popen([sys.executable,'-m','scripts.control_uio66_sampling_pilot',
         '--checkpoint',str(CHECKPOINT),'--bundle',str(BUNDLE),'--bundle-sha256',AUTH['bundle_sha256'],
         '--allocation-start-utc',ALLOCATION_START_UTC,'--output',str(OUT)],env=ENV,
         stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    print('Controller launched:',CONTROLLER.pid,'| Deadline:',ALLOCATION_START_UTC,'+4.2h')
    print('Chemistry stop: write a specific reason to',OUT/'STOP')

    CONTROLLER.wait()
    print('CONTROLLER TERMINAL:',CONTROLLER.returncode,flush=True)
    from collections import Counter
    print('Controller exit:',CONTROLLER.poll())
    if (OUT/'allocation.json').exists():
        allocation=json.loads((OUT/'allocation.json').read_text())
        print('STATUS:',allocation['status'],'| DEADLINE:',allocation['deadline_utc'])
        print('STOP:',allocation.get('stop_reason'))
    for i in range(4):
        report=OUT/f'worker{i}'/'adsorption.json'
        if report.exists():
            data=json.loads(report.read_text())
            active=data.get('active_candidate') or {}
            completed=[s for r in data['results'] for s in r.get('starts',[])]+active.get('starts',[])
            print(i,'finished designs',len(data['results']),'active',active.get('id'),
                  'guest attempts',len(completed),'statuses',dict(Counter(s['status'] for s in completed)))
    print(controller_log.read_text()[-1600:] if controller_log.exists() else '')

    workflow['status']='controller returned; inspect allocation and frozen assessment'
except BaseException as error:
    workflow.update(status='failed; partial evidence retained',error_type=type(error).__name__,error=str(error))
    traceback.print_exc()
finally:
    controller=globals().get('CONTROLLER')
    if controller is not None and controller.poll() is None:
        raise RuntimeError('Owned controller still live; do not archive live trajectories as final')
    workflow['finished_utc']=datetime.now(timezone.utc).isoformat()
    state_path=WORKING/'saved_batch_state.json'
    state_path.write_text(json.dumps(workflow,indent=2)+chr(10))
    evidence=WORKING/'UIO66_sampling_saved_batch_evidence.zip'
    assert not evidence.exists(), 'Never overwrite evidence'
    with zipfile.ZipFile(evidence,'w',zipfile.ZIP_DEFLATED) as archive:
        archive.write(state_path,state_path.name)
        for folder in (globals().get('OUT'),globals().get('PRE')):
            if folder is not None and folder.exists():
                for file in sorted(folder.rglob('*')):
                    if file.is_file(): archive.write(file,str(file.relative_to(WORKING)))
        log=globals().get('controller_log')
        if log is not None and log.exists(): archive.write(log,log.name)
    print('SAVED BATCH STATE:',json.dumps(workflow),flush=True)
    print('EVIDENCE:',evidence.name,evidence.stat().st_size,hashlib.sha256(evidence.read_bytes()).hexdigest(),flush=True)
